# 04 — Evaluation

Runs automated intent/escalation metrics on the golden set and provides an LLM-judge hook. Golden labels must be reviewed by a human before they are described as hand-labelled.

In [ ]:
import sys,os
from pathlib import Path
import pandas as pd
from sklearn.metrics import accuracy_score,f1_score,precision_recall_fscore_support
sys.path.insert(0,str(Path('..').resolve()))
from src.retrieval import Retriever
from src.model import IntentModel
from src.agent import escalate,reply
DATA=Path('../data')
if not DATA.exists(): DATA=Path('data')
g=pd.read_csv(DATA/'golden_eval_set.csv').fillna('')
h=pd.read_csv(DATA/'historical_support_pairs.csv').fillna('')
r=Retriever(h); m=IntentModel(); m.fit(h.clean_message,h.conversation_id)


In [ ]:
def evaluate_row(row):
    intent,conf=m.predict(row.clean_message)
    ev=r.search(row.clean_message,k=3,exclude_conversation_id=row.conversation_id)
    sim=ev[0]['similarity'] if ev else 0
    esc,reason=escalate(row.clean_message,sim,conf)
    draft=reply(row.clean_message,intent,ev,os.getenv('OPENAI_API_KEY'),os.getenv('OPENAI_MODEL','gpt-4o-mini'))
    return intent,conf,sim,esc,reason,draft

out=[]
for _,row in g.iterrows():
    vals=evaluate_row(row); out.append(vals)
for i,c in enumerate(['predicted_intent','confidence','top_similarity','predicted_escalation','escalation_reason','reply']): g[c]=[x[i] for x in out]
g.to_csv(DATA/'agent_outputs.csv',index=False)


In [ ]:
valid=g[g.golden_intent.str.strip()!='']
if len(valid):
    print('Golden intent accuracy:',accuracy_score(valid.golden_intent,valid.predicted_intent))
    print('Golden intent macro-F1:',f1_score(valid.golden_intent,valid.predicted_intent,average='macro',zero_division=0))
else: print('golden_intent is blank; review labels before reporting golden metrics.')
valid_e=g[g.golden_escalation.str.strip()!='']
if len(valid_e):
    yt=valid_e.golden_escalation.str.lower().isin(['yes','true','1']); yp=valid_e.predicted_escalation
    p,r_,f,_=precision_recall_fscore_support(yt,yp,average='binary',zero_division=0)
    print({'precision':p,'recall':r_,'f1':f})
else: print('golden_escalation is blank; review labels before reporting escalation metrics.')
